# Citizen Service Root-Cause Analysis
**Important:** No original CSV was supplied. This notebook creates a reproducible synthetic dataset of 800,000 rows. All results are illustrative and must not be presented as findings from a real Pune government system.

In [ ]:
# Citizen Service Root-Cause Analysis
# Synthetic teaching dataset: 800,000 service requests; no source CSV was supplied.
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

rng=np.random.default_rng(42)
n=800_000
quarter=np.repeat(["Q1","Q2"],n//2)
services=["Certificates","Scholarship","Property Tax","Licences","Grievances"]
p={"Q1":[.22,.16,.24,.20,.18],"Q2":[.28,.22,.18,.16,.16]}
service=np.empty(n,dtype=object)
for q in ["Q1","Q2"]:
    m=quarter==q
    service[m]=rng.choice(services,m.sum(),p=p[q])
office=np.array(["PUN01","PUN02","PUN03","PUN04","PUN05","PUN06"])
office=np.random.default_rng(43).choice(office,n,p=[.19,.18,.17,.16,.15,.15])
district=rng.choice(["Central","East","West","North","South"],n,p=[.24,.20,.20,.18,.18])
citizen=rng.integers(1,260001,n)
mobile_suffix=rng.integers(0,3,n)
mobile=[f"9{(int(c)*7919+int(m)*104729)%10**9:09d}" for c,m in zip(citizen,mobile_suffix)]
week=np.where(quarter=="Q1",rng.integers(1,14,n),rng.integers(14,27,n))
deadline=np.isin(week,[7,13,20,26])
raw_office=office.copy()
raw_office[(quarter=="Q2")&(office=="PUN05")]="PN05A"
raw_office[(quarter=="Q2")&(office=="PUN06")]="PN06B"
complex_service=np.isin(service,["Certificates","Scholarship"])
base=np.select([service=="Certificates",service=="Scholarship",service=="Property Tax",service=="Licences"],[40,46,24,30],default=28).astype(float)
tat=np.clip(base+(quarter=="Q2")*9+complex_service*8+np.isin(office,["PUN05","PUN06"])*(quarter=="Q2")*18+deadline*12+rng.normal(0,15,n),2,240)
sla=np.where(complex_service,48,36)
df=pd.DataFrame({"citizen_id":citizen,"mobile":mobile,"service_type":service,"office_code_raw":raw_office,"district":district,"week":week,"quarter":quarter,"turnaround_hours":tat,"sla_hours":sla,"status_raw":np.where(tat<=sla,"Resolved","Pending"),"deadline_period":deadline})
# Canonical office mapping fixes mid-year code changes.
office_map={"PN05A":"PUN05","PN06B":"PUN06"}
df["office_id"]=df.office_code_raw.replace(office_map)
df["status_clean"]=df.status_raw.astype(str).str.strip().str.lower()
df["within_sla"]=(df.turnaround_hours<=df.sla_hours).astype(int)
df["citizen_segment"]=np.where(df.citizen_id%5==0,"Repeat citizen","Single/occasional")

# Metric decomposition
print(df.groupby("quarter").within_sla.agg(["mean","count"]))
print(pd.crosstab(df.quarter,df.within_sla))
print(df.groupby(["quarter","service_type"]).within_sla.mean().unstack())
print(df.groupby(["quarter","office_id"]).within_sla.mean().unstack())
# Tests: Welch t-test on turnaround time; chi-square on within-SLA outcome.
a=df.loc[df.quarter=="Q1","turnaround_hours"].sample(50000,random_state=1)
b=df.loc[df.quarter=="Q2","turnaround_hours"].sample(50000,random_state=2)
print("Welch t-test:",stats.ttest_ind(a,b,equal_var=False))
print("Chi-square:",stats.chi2_contingency(pd.crosstab(df.quarter,df.within_sla)))
# Deadline sensitivity
print(df.groupby(["quarter","deadline_period"]).within_sla.mean())
# Citizen-level robustness check: aggregate multiple requests per citizen.
print(df.groupby(["quarter","citizen_id"]).within_sla.mean().groupby("quarter").mean())
